In [0]:
catalog = dbutils.widgets.get('catalog')
schema_bronze = dbutils.widgets.get('schema_bronze')
schema_silver = dbutils.widgets.get('schema_silver')
table_bronze = dbutils.widgets.get('table_bronze')
table_silver = dbutils.widgets.get('table_silver')



In [0]:
from pyspark.sql import functions as sf 
from pyspark.sql.dataframe import DataFrame
df_bronze = spark.read.table(f'{catalog}.{schema_bronze}.{table_bronze}')




## Transform Data



In [0]:
df_bronze = df_bronze.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
)

data_type_mapping = {
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string",
    "order_purchase_timestamp": "timestamp",
    "order_approved_at": "timestamp",
    "order_delivered_carrier_date": "timestamp",
    "order_delivered_customer_date": "timestamp",
    "order_estimated_delivery_date": "timestamp",
}


def cast_columns(df: DataFrame, mapping: dict) -> DataFrame:
    existing_columns = df.columns

    for column, data_type in mapping.items():
        if column not in existing_columns:
            raise ValueError(f"Column '{column}' not found in DataFrame")
            

        df = df.withColumn(column, sf.col(column).cast(data_type))
        print(f"Column '{column}' casted to {data_type}")
    
    return df

df_bronze = cast_columns(df_bronze, data_type_mapping)
        



In [0]:


def standardardizing_column_name(df: DataFrame) -> DataFrame:
    new_columns = [column.strip().replace(" ", "_").lower() for column in df.columns]
    return df.toDF(*new_columns)

def standardarizing_raw(df: DataFrame) -> DataFrame:
    string_columns = [col_name for col_name, dtype in df.dtypes if dtype == 'string']

    for col in string_columns:
        df = df.withColumn(col, sf.trim(sf.lower(sf.col(col))))
        
    return df
    

df_bronze_column_normalized = standardardizing_column_name(df_bronze)
df_bronze_standardized = standardarizing_raw(df_bronze_column_normalized)


In [0]:
df_silver = df_bronze_standardized.withColumn("silver_update_date", sf.current_timestamp())

In [0]:
%run ../utils/utils_merge_into_tables

upsert_data(df_silver, table_silver,primary_keys) # noqa: F821